<p align="center">
  <img src="https://img.shields.io/badge/Project-Multi--Class&nbsp;Brain&nbsp;Tumor&nbsp;MRI-blue" />
  <img src="https://img.shields.io/badge/Pipeline-Custom&nbsp;CNN&nbsp;vs.&nbsp;Transfer&nbsp;Learning-green" />
</p>

# 🎓 RQ5 Notebook · *Custom CNN* **vs.** *Transfer‑Learning ResNet‑18*  (✅ _Fixed_)

<sub>This version resolves channel‑mismatch issues so both **SimpleCNN/LpCNN** and **ResNet‑18** train on the same 3‑channel transform.</sub>

*Generated {now}*


In [1]:

import time, os, copy, random, sys
from collections import defaultdict

import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report

# Project modules
sys.path.append(os.path.abspath(".."))
from src.dataset import BrainTumorDataset
from src.transforms_config import create_transform
from src.models import SimpleCNN
print('Torch', torch.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device →', DEVICE)


Torch 2.7.0+cpu
Device → cpu


In [2]:

DATA_ROOT    = '../cleaned_data'   # path to dataset
INPUT_SIZE   = 160
BATCH_SIZE   = 16
N_EPOCHS     = 10
LR           = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.0
NUM_CLASSES  = 4

# 3‑channel transform (one_ch_to_three_ch=True)
TRANSFORM = create_transform(
    image_size=(INPUT_SIZE, INPUT_SIZE),
    augment=False,
    adv_aug=False,
    strong_aug=False,
    one_ch_to_three_ch=True
)


In [3]:

train_set = BrainTumorDataset(DATA_ROOT, subset='Training', transform=TRANSFORM, model_variant='simple')
val_set   = BrainTumorDataset(DATA_ROOT, subset='Testing',  transform=TRANSFORM, model_variant='simple')

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train samples: {len(train_set)} | Val samples: {len(val_set)}')


Train samples: 5442 | Val samples: 1284


In [4]:

def train_one_epoch(model, loader, criterion, optimizer):
    model.train(); running_loss=0.0; running_corrects=0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        running_loss += loss.item()*x.size(0)
        running_corrects += (logits.argmax(1)==y).sum().item()
    return running_loss/len(loader.dataset), running_corrects/len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); loss=0.0; corr=0; y_true=[]; y_pred=[]
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x); loss += criterion(logits, y).item()*x.size(0)
        preds = logits.argmax(1)
        corr += (preds==y).sum().item()
        y_true.extend(y.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
    acc = corr/len(loader.dataset); f1 = f1_score(y_true, y_pred, average='macro')
    return loss/len(loader.dataset), acc, f1, y_true, y_pred


In [5]:

# --- SimpleCNN (patched for 3‑channel input) ---
custom_model = SimpleCNN(
    num_classes=NUM_CLASSES,
    input_size=INPUT_SIZE,
    dropout=DROPOUT
).to(DEVICE)

# If the first conv expects 1 channel, swap it for 3‑channel version
first_conv = None
for name, module in custom_model.named_modules():
    if isinstance(module, nn.Conv2d):
        first_conv = module
        break
if first_conv is not None and first_conv.in_channels == 1:
    print('Patching SimpleCNN first conv: 1‑→3 channels')
    new_conv = nn.Conv2d(
        in_channels=3,
        out_channels=first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=first_conv.bias is not None
    )
    custom_model.features[0] = new_conv  # assumes Sequential
else:
    print('SimpleCNN already compatible.')

crit = nn.CrossEntropyLoss()
opt  = optim.Adam(custom_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_acc = 0.0
for ep in range(N_EPOCHS):
    train_one_epoch(custom_model, train_loader, crit, opt)
    _, v_acc, v_f1, _, _ = evaluate(custom_model, val_loader, crit)
    print(f'[Custom] {ep+1:02d}/{N_EPOCHS}  Val acc {v_acc*100:5.2f}%  F1 {v_f1:.3f}')
    best_acc = max(best_acc, v_acc)
print(f'Best CustomCNN accuracy: {best_acc*100:.2f}%')


Patching SimpleCNN first conv: 1‑→3 channels


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Custom] 01/10  Val acc 66.82%  F1 0.626


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Custom] 02/10  Val acc 77.57%  F1 0.750


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


... (11 outputs left out)

C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Custom] 09/10  Val acc 92.99%  F1 0.928


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Custom] 10/10  Val acc 92.13%  F1 0.918
Best CustomCNN accuracy: 92.99%


In [ ]:

import torchvision.models as models
resnet = models.resnet18(weights='IMAGENET1K_V1')
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = resnet.to(DEVICE)

# Freeze backbone for 1 epoch
for p in resnet.parameters(): p.requires_grad = False
for p in resnet.fc.parameters(): p.requires_grad = True

crit_tl = nn.CrossEntropyLoss()
opt_tl  = optim.Adam(resnet.fc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_tl = 0.0
for ep in range(N_EPOCHS):
    if ep == 1:
        for p in resnet.parameters(): p.requires_grad = True
        opt_tl = optim.Adam(resnet.parameters(), lr=LR*0.1, weight_decay=WEIGHT_DECAY)
    train_one_epoch(resnet, train_loader, crit_tl, opt_tl)
    _, v_acc, v_f1, _, _ = evaluate(resnet, val_loader, crit_tl)
    print(f'[ResNet18] {ep+1:02d}/{N_EPOCHS}  Val acc {v_acc*100:5.2f}%  F1 {v_f1:.3f}')
    best_tl = max(best_tl, v_acc)
print(f'Best ResNet‑18 accuracy: {best_tl*100:.2f}%')


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[ResNet18] 01/10  Val acc 78.89%  F1 0.779


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[ResNet18] 02/10  Val acc 95.02%  F1 0.949


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[ResNet18] 03/10  Val acc 95.40%  F1 0.953


... (3 outputs left out)

[ResNet18] 05/10  Val acc 98.05%  F1 0.980


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[ResNet18] 06/10  Val acc 97.59%  F1 0.975


C:\Users\yannk\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:

from src.lpconv import LpCNN
lpcnn = LpCNN(NUM_CLASSES, INPUT_SIZE, DROPOUT, in_channels=3).to(DEVICE)   # in_channels=3
opt_lp = optim.Adam(lpcnn.parameters(), lr=LR)
crit_lp = nn.CrossEntropyLoss()

best_lp = 0.0
for ep in range(N_EPOCHS):
    train_one_epoch(lpcnn, train_loader, crit_lp, opt_lp)
    _, acc, f1, _, _ = evaluate(lpcnn, val_loader, crit_lp)
    best_lp = max(best_lp, acc)
    print(f'[LpCNN] {ep+1:02d}/{N_EPOCHS}  Val acc {acc*100:.2f}%  F1 {f1:.3f}')
print(f'Best LpCNN accuracy: {best_lp*100:.2f}%')
